# Empório da Música — Exploração e validação dos dados

**Objetivo:** explorar os seis conjuntos de dados CSV fornecidos, validar a qualidade dos dados e a integridade dos relacionamentos, além de documentar os resultados antes da implementação da camada de dados da aplicação.

Desenvolvido para **Google Colab**. Este notebook não modifica os arquivos CSV de origem.

### Escopo
- Carregar e analisar categorias, clientes, produtos, pedidos, itens de pedidos e promoções.
- Inspecionar tipos de dados, valores ausentes, duplicatas e distribuições.
- Validar possíveis chaves primárias e relacionamentos de chaves estrangeiras.
- Executar verificações selecionadas de domínio e datas.
- Exportar relatórios de validação para a documentação do projeto.

## 1. Configuração do ambiente


In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
print("pandas:", pd.__version__)

pandas: 2.2.3


## 2. Carregar os seis arquivos CSV de origem

Envie os seis arquivos CSV do desafio. A identificação dos nomes de arquivo não diferencia maiúsculas de minúsculas e procura o nome do conjunto de dados em qualquer parte do nome do arquivo.

In [4]:
from google.colab import files

uploaded = files.upload()
uploaded_names = list(uploaded.keys())
print("Uploaded:", uploaded_names)

Saving desafio_tecnico_ai_eng - categories.csv to desafio_tecnico_ai_eng - categories.csv
Saving desafio_tecnico_ai_eng - customers.csv to desafio_tecnico_ai_eng - customers.csv
Saving desafio_tecnico_ai_eng - order_items.csv to desafio_tecnico_ai_eng - order_items.csv
Saving desafio_tecnico_ai_eng - orders.csv to desafio_tecnico_ai_eng - orders.csv
Saving desafio_tecnico_ai_eng - products.csv to desafio_tecnico_ai_eng - products.csv
Saving desafio_tecnico_ai_eng - promotions.csv to desafio_tecnico_ai_eng - promotions.csv
Uploaded: ['desafio_tecnico_ai_eng - categories.csv', 'desafio_tecnico_ai_eng - customers.csv', 'desafio_tecnico_ai_eng - order_items.csv', 'desafio_tecnico_ai_eng - orders.csv', 'desafio_tecnico_ai_eng - products.csv', 'desafio_tecnico_ai_eng - promotions.csv']


In [5]:
DATASET_KEYWORDS = {
    "categories": "categories",
    "customers": "customers",
    "products": "products",
    "orders": "orders",
    "order_items": "order_items",
    "promotions": "promotions",
}

file_map = {}
for dataset, keyword in DATASET_KEYWORDS.items():
    matches = [name for name in uploaded_names if keyword in name.lower()]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one file for {dataset}; found: {matches}")
    file_map[dataset] = matches[0]

data = {name: pd.read_csv(filename) for name, filename in file_map.items()}

for name, df in data.items():
    print(f"{name}: {df.shape[0]} rows × {df.shape[1]} columns")

categories: 9 rows × 3 columns
customers: 50 rows × 5 columns
products: 65 rows × 9 columns
orders: 20 rows × 9 columns
order_items: 22 rows × 3 columns
promotions: 25 rows × 5 columns


## 3. Inventário inicial

In [6]:
inventory = pd.DataFrame([
    {"dataset": name, "rows": len(df), "columns": len(df.columns),
     "column_names": ", ".join(map(str, df.columns))}
    for name, df in data.items()
]).sort_values("dataset")
display(inventory)

for name, df in data.items():
    print(f"\n### {name}")
    display(df.head())
    display(df.dtypes.rename("dtype").to_frame())

,dataset,rows,columns,column_names
0,categories,9,3,"category_id, name, description"
1,customers,50,5,"customer_id, name, phone, email, city"
4,order_items,22,3,"order_id, quantity, product_id"
3,orders,20,9,"order_id, customer_id, order_date, status, total_brl, payment_method, tracking_code, estimated_delivery, notes"
2,products,65,9,"product_id, price_brl, name, category_id, description, stock_quantity, status, specs, created_at"
5,promotions,25,5,"promotion_id, product_id, discount_percent, description, is_active"



### categories


,category_id,name,description
0,1,Guitarras,"Guitarras elétricas de todos os estilos, do rock ao jazz"
1,2,Baixos,Baixos elétricos e acústicos para todas as formações
2,3,Baterias e Percussão,"Baterias acústicas, eletrônicas e peças avulsas de percussão"
3,4,Teclados e Pianos,"Pianos digitais, sintetizadores, arranjadores e controladores MIDI"
4,5,Violões,"Violões clássicos, folk e eletroacústicos"


,dtype
category_id,int64
name,object
description,object



### customers


,customer_id,name,phone,email,city
0,1,Lucas Mendes da Silva,(67) 99812-3456,lucas.mendes@jmail.com,Campo Grande
1,2,Ana Carolina Ferreira,(67) 99654-7821,anacarol.ferreira@coldmail.com,Campo Grande
2,3,Pedro Henrique Oliveira,(67) 98432-1098,pedro.oliveira@jmail.com,Campo Grande
3,4,Mariana Costa Santos,(67) 99123-4567,mariana.costa@inlook.com,Campo Grande
4,5,Rafael Augusto Pereira,(67) 98765-4321,rafael.pereira@jmail.com,Campo Grande


... (9 outputs left out)

,dtype
order_id,int64
quantity,int64
product_id,int64



### promotions


,promotion_id,product_id,discount_percent,description,is_active
0,1,102,10,Black Friday,0
1,2,110,15,Liquidação de Inverno,0
2,3,122,20,Queima de Estoque,0
3,4,95,12,Oferta Especial,0
4,5,126,25,Aniversário da Loja,0


,dtype
promotion_id,int64
product_id,int64
discount_percent,int64
description,object
is_active,int64


## 4. Perfil de qualidade dos dados por coluna

In [7]:
profile_rows = []
for dataset, df in data.items():
    for column in df.columns:
        s = df[column]
        profile_rows.append({
            "dataset": dataset,
            "column": column,
            "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()),
            "missing_pct": round(float(s.isna().mean() * 100), 2),
            "distinct_count": int(s.nunique(dropna=True)),
            "sample_values": s.dropna().astype(str).drop_duplicates().head(5).tolist(),
        })
column_profile = pd.DataFrame(profile_rows)
display(column_profile)

duplicate_summary = pd.DataFrame([
    {"dataset": name, "exact_duplicate_rows": int(df.duplicated().sum())}
    for name, df in data.items()
])
display(duplicate_summary)

,dataset,column,dtype,missing_count,missing_pct,distinct_count,sample_values
0,categories,category_id,int64,0,0.0,9,"[1, 2, 3, 4, 5]"
1,categories,name,object,0,0.0,9,"[Guitarras, Baixos, Baterias e Percussão, Teclados e Pianos, Violões]"
2,categories,description,object,0,0.0,9,"[Guitarras elétricas de todos os estilos, do rock ao jazz, Baixos elétricos e acústicos para todas as formações, Bat..."
3,customers,customer_id,int64,0,0.0,50,"[1, 2, 3, 4, 5]"
4,customers,name,object,0,0.0,50,"[Lucas Mendes da Silva, Ana Carolina Ferreira, Pedro Henrique Oliveira, Mariana Costa Santos, Rafael Augusto Pereira]"
5,customers,phone,object,0,0.0,50,"[(67) 99812-3456, (67) 99654-7821, (67) 98432-1098, (67) 99123-4567, (67) 98765-4321]"
6,customers,email,object,0,0.0,49,"[lucas.mendes@jmail.com, anacarol.ferreira@coldmail.com, pedro.oliveira@jmail.com, mariana.costa@inlook.com, rafael...."
7,customers,city,object,0,0.0,5,"[Campo Grande, Dourados, Três Lagoas, Corumbá, Ponta Porã]"
8,products,product_id,int64,0,0.0,65,"[81, 82, 83, 84, 85]"
9,products,price_brl,float64,0,0.0,58,"[599.9, 849.0, 429.9, 689.0, 459.9]"


,dataset,exact_duplicate_rows
0,categories,0
1,customers,0
2,products,0
3,orders,0
4,order_items,0
5,promotions,0


## 5. Validação de possíveis chaves primárias


In [8]:
PRIMARY_KEYS = {
    "categories": ["category_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "promotions": ["promotion_id"],
}
pk_rows = []
for table, cols in PRIMARY_KEYS.items():
    df = data[table]
    exists = all(c in df.columns for c in cols)
    if not exists:
        pk_rows.append({"table": table, "key": ", ".join(cols), "columns_exist": False,
                        "null_key_rows": None, "duplicate_key_rows": None, "valid": False})
        continue
    nulls = int(df[cols].isna().any(axis=1).sum())
    duplicates = int(df.duplicated(subset=cols, keep=False).sum())
    pk_rows.append({"table": table, "key": ", ".join(cols), "columns_exist": True,
                    "null_key_rows": nulls, "duplicate_key_rows": duplicates,
                    "valid": nulls == 0 and duplicates == 0})
pk_report = pd.DataFrame(pk_rows)
display(pk_report)

,table,key,columns_exist,null_key_rows,duplicate_key_rows,valid
0,categories,category_id,True,0,0,True
1,customers,customer_id,True,0,0,True
2,products,product_id,True,0,0,True
3,orders,order_id,True,0,0,True
4,promotions,promotion_id,True,0,0,True


## 6. Validação de chaves estrangeiras e integridade referencial

Relacionamentos verificados:
- `products.category_id` → `categories.category_id`
- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`
- `promotions.product_id` → `products.product_id`

Valores nulos em chaves estrangeiras e referências órfãs são reportados separadamente.

In [9]:
FOREIGN_KEYS = [
    ("products", "category_id", "categories", "category_id"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("promotions", "product_id", "products", "product_id"),
]
fk_rows = []
for child_t, child_c, parent_t, parent_c in FOREIGN_KEYS:
    child, parent = data[child_t], data[parent_t]
    nulls = int(child[child_c].isna().sum())
    parent_values = set(parent[parent_c].dropna())
    orphans = int((~child[child_c].dropna().isin(parent_values)).sum())
    fk_rows.append({
        "child_table": child_t, "child_column": child_c,
        "parent_table": parent_t, "parent_column": parent_c,
        "null_fk_count": nulls, "orphan_count": orphans,
        "valid_no_orphans": orphans == 0
    })
fk_report = pd.DataFrame(fk_rows)
display(fk_report)

,child_table,child_column,parent_table,parent_column,null_fk_count,orphan_count,valid_no_orphans
0,products,category_id,categories,category_id,0,0,True
1,orders,customer_id,customers,customer_id,0,0,True
2,order_items,order_id,orders,order_id,0,0,True
3,order_items,product_id,products,product_id,0,0,True
4,promotions,product_id,products,product_id,0,0,True


## 7. Verificações de domínio e consistência das regras de negócio

In [10]:
domain_rows = []

def add_check(name, invalid_count, note=""):
    domain_rows.append({"check": name, "invalid_rows": int(invalid_count),
                        "passed": int(invalid_count) == 0, "note": note})

p = data["products"]
if "price_brl" in p:
    add_check("Product price > 0", (p["price_brl"].notna() & (p["price_brl"] <= 0)).sum())
if "stock_quantity" in p:
    add_check("Product stock >= 0", (p["stock_quantity"].notna() & (p["stock_quantity"] < 0)).sum())

oi = data["order_items"]
if "quantity" in oi:
    add_check("Order item quantity > 0", (oi["quantity"].notna() & (oi["quantity"] <= 0)).sum())

promo = data["promotions"]
if "discount_percent" in promo:
    invalid = promo["discount_percent"].notna() & ~promo["discount_percent"].between(0, 100)
    add_check("Promotion discount between 0 and 100", invalid.sum())

if {"order_id", "product_id"}.issubset(oi.columns):
    dup_pairs = oi.duplicated(["order_id", "product_id"], keep=False).sum()
    add_check("Order/product pairs unique in supplied rows", dup_pairs,
              "Confirm whether one product may appear on multiple lines in one order.")

if "specs" in p:
    def valid_json(value):
        if pd.isna(value) or str(value).strip() == "":
            return True
        try:
            json.loads(value)
            return True
        except (TypeError, json.JSONDecodeError):
            return False
    add_check("Product specs valid JSON when populated", (~p["specs"].apply(valid_json)).sum())

domain_report = pd.DataFrame(domain_rows)
display(domain_report)

for table, column in [("products", "status"), ("orders", "status"),
                      ("orders", "payment_method"), ("promotions", "is_active")]:
    if column in data[table]:
        print(f"\n{table}.{column}")
        display(data[table][column].value_counts(dropna=False).rename("count").to_frame())

,check,invalid_rows,passed,note
0,Product price > 0,0,True,
1,Product stock >= 0,0,True,
2,Order item quantity > 0,0,True,
3,Promotion discount between 0 and 100,0,True,
4,Order/product pairs unique in supplied rows,0,True,Confirm whether one product may appear on multiple lines in one order.
5,Product specs valid JSON when populated,0,True,



products.status


,count
status,
active,62
discontinued,2
coming_soon,1



orders.status


,count
status,
delivered,7
confirmed,4
pending,4
cancelled,4
shipped,1



orders.payment_method


,count
payment_method,
pix,5
credit_12x,4
credit_6x,3
boleto,3
credit_3x,3
debit,2



promotions.is_active


,count
is_active,
0,21
1,4


## 8. Investigação de possíveis ambiguidades nos dados

### 8.1 Unicidade do e-mail dos clientes



In [11]:
customers = data["customers"]
if "email" in customers:
    duplicated_emails = customers[
        customers["email"].notna() & customers["email"].duplicated(keep=False)
    ].sort_values("email")
    print("Rows with duplicated non-null emails:", len(duplicated_emails))
    display(duplicated_emails)

Rows with duplicated non-null emails: 2


,customer_id,name,phone,email,city
10,11,Bruno Carvalho Martins,(67) 98543-2109,bruno.martins@jmail.com,Três Lagoas
42,43,Bruno Martins,(67) 99456-3456,bruno.martins@jmail.com,Três Lagoas


### 8.2 Múltiplos registros de promoção por produto


In [12]:
if {"product_id", "promotion_id"}.issubset(promo.columns):
    display(
        promo.groupby("product_id").size().reset_index(name="promotion_rows")
        .query("promotion_rows > 1").sort_values("promotion_rows", ascending=False)
    )
if {"product_id", "is_active"}.issubset(promo.columns):
    active = promo[promo["is_active"].astype(str).str.lower().isin(["true", "1", "yes"])]
    print("Products with multiple active promotions:")
    display(active.groupby("product_id").size().reset_index(name="active_promotion_rows")
            .query("active_promotion_rows > 1"))

,product_id,promotion_rows
2,93,2
5,96,2
6,99,2


Products with multiple active promotions:


,product_id,active_promotion_rows


## 9. Conversão de datas e informações operacionais ausentes


In [13]:
DATE_COLUMNS = {
    "products": ["created_at"],
    "orders": ["order_date", "estimated_delivery"],
}
date_rows = []
for table, columns in DATE_COLUMNS.items():
    for col in columns:
        if col not in data[table]:
            date_rows.append({"table": table, "column": col, "exists": False,
                              "non_null_values": 0, "parse_failures": None})
            continue
        raw = data[table][col]
        parsed = pd.to_datetime(raw, errors="coerce")
        failures = int((raw.notna() & parsed.isna()).sum())
        date_rows.append({"table": table, "column": col, "exists": True,
                          "non_null_values": int(raw.notna().sum()),
                          "parse_failures": failures})
date_report = pd.DataFrame(date_rows)
display(date_report)

for col in ["tracking_code", "estimated_delivery", "notes"]:
    if col in data["orders"]:
        print(f"orders.{col}: {int(data['orders'][col].isna().sum())} null values")

,table,column,exists,non_null_values,parse_failures
0,products,created_at,True,65,0
1,orders,order_date,True,20,0
2,orders,estimated_delivery,True,8,0


orders.tracking_code: 12 null values
orders.estimated_delivery: 12 null values
orders.notes: 16 null values


## 10. Resumo consolidado da validação

In [14]:
summary = pd.Series({
    "datasets_loaded": len(data),
    "total_rows": sum(len(df) for df in data.values()),
    "total_columns": sum(len(df.columns) for df in data.values()),
    "exact_duplicate_rows": sum(int(df.duplicated().sum()) for df in data.values()),
    "primary_key_checks_passed": int(pk_report["valid"].sum()),
    "primary_key_checks_total": len(pk_report),
    "foreign_key_checks_without_orphans": int(fk_report["valid_no_orphans"].sum()),
    "foreign_key_checks_total": len(fk_report),
    "domain_checks_passed": int(domain_report["passed"].sum()),
    "domain_checks_total": len(domain_report),
}, name="value")
display(summary.to_frame())

print("Primary-key issues")
display(pk_report[~pk_report["valid"]])
print("Foreign-key issues")
display(fk_report[~fk_report["valid_no_orphans"]])
print("Domain-check issues")
display(domain_report[~domain_report["passed"]])

,value
datasets_loaded,6
total_rows,191
total_columns,34
exact_duplicate_rows,0
primary_key_checks_passed,5
primary_key_checks_total,5
foreign_key_checks_without_orphans,5
foreign_key_checks_total,5
domain_checks_passed,6
domain_checks_total,6


Primary-key issues


,table,key,columns_exist,null_key_rows,duplicate_key_rows,valid


Foreign-key issues


,child_table,child_column,parent_table,parent_column,null_fk_count,orphan_count,valid_no_orphans


Domain-check issues


,check,invalid_rows,passed,note


## 11. Resultados, premissas e decisões de engenharia

Esta seção resume os resultados observados quando este notebook foi executado sobre os seis arquivos CSV fornecidos.

### 11.1 Visão geral dos conjuntos de dados

Os seis conjuntos de dados contêm **191 linhas e 34 colunas** no total.

| Conjunto de dados | Linhas | Colunas |
|---|---:|---:|
| categories | 9 | 3 |
| customers | 50 | 5 |
| products | 65 | 9 |
| orders | 20 | 9 |
| order_items | 22 | 3 |
| promotions | 25 | 5 |
| **Total** | **191** | **34** |

Nenhuma linha exatamente duplicada foi encontrada em qualquer um dos seis conjuntos de dados.

### 11.2 Integridade estrutural

Todas as cinco verificações de possíveis chaves primárias foram aprovadas:

- `categories.category_id`
- `customers.customer_id`
- `products.product_id`
- `orders.order_id`
- `promotions.promotion_id`

Para cada chave candidata, o notebook encontrou zero valores nulos e zero linhas com chaves duplicadas.

Todas as cinco verificações de relacionamento por chave estrangeira reportaram zero referências órfãs e zero chaves estrangeiras nulas:

- `products.category_id` → `categories.category_id`
- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`
- `promotions.product_id` → `products.product_id`

Esses resultados dão suporte aos relacionamentos propostos para a amostra fornecida.

### 11.3 Verificações de domínio

Todas as seis verificações de domínio implementadas foram aprovadas, com **zero linhas inválidas**:

| Verificação | Linhas inválidas | Resultado |
|---|---:|---|
| O preço do produto é maior que zero | 0 | Aprovado |
| A quantidade em estoque do produto é não negativa | 0 | Aprovado |
| A quantidade do item do pedido é maior que zero | 0 | Aprovado |
| O desconto da promoção está entre 0 e 100 | 0 | Aprovado |
| Os pares `(order_id, product_id)` são únicos nas linhas fornecidas | 0 | Aprovado na amostra |
| Os valores preenchidos de `specs` dos produtos contêm JSON válido | 0 | Aprovado |

A unicidade de `(order_id, product_id)` é uma propriedade observada nesta amostra, não uma restrição de negócio confirmada. Confirme se o mesmo produto pode aparecer em várias linhas de um pedido antes de adotar esse par como chave primária permanente de `order_items`.

As verificações acima não estabelecem que todas as colunas sejam não nulas nem que todo valor categórico pertença a um domínio aprovado. Isso exige regras e validações específicas.

### 11.4 Valores ausentes

O perfil por coluna encontrou valores ausentes somente nos seguintes campos de pedidos:

| Coluna | Linhas ausentes | Percentual ausente |
|---|---:|---:|
| `tracking_code` | 12 de 20 | 60% |
| `estimated_delivery` | 12 de 20 | 60% |
| `notes` | 16 de 20 | 80% |

Todas as demais colunas dos conjuntos de dados fornecidos apresentaram zero valores ausentes nesta execução.

Códigos de rastreamento e datas estimadas de entrega ausentes devem ser tratados como informações indisponíveis. O agente de atendimento não deve inventar nenhum desses valores. Observações ausentes devem ser tratadas como um campo opcional não preenchido, e não como evidência de uma condição específica do pedido.

### 11.5 Distribuições categóricas

**Status dos produtos (65 produtos):**

- `active`: 62
- `discontinued`: 2
- `coming_soon`: 1

A aplicação deve distinguir o status no catálogo da disponibilidade em estoque. Um produto marcado como `active` não está necessariamente disponível se sua quantidade atual em estoque for zero. A análise anterior dos dados identificou produtos sem estoque; o Repository deve verificar tanto o status quanto a quantidade em estoque.

**Status dos pedidos (20 pedidos):**

- `delivered`: 7
- `confirmed`: 4
- `pending`: 4
- `cancelled`: 4
- `shipped`: 1

Esses valores fornecem exemplos para testes de resposta sobre status de pedidos. Essa distribuição de frequências é descritiva e não deve ser interpretada como uma medida representativa da operação da loja.

**Forma de pagamento (20 pedidos):**

- `pix`: 5
- `credit_12x`: 4
- `credit_6x`: 3
- `boleto`: 3
- `credit_3x`: 3
- `debit`: 2

Essas são as formas de pagamento presentes na amostra histórica. Sua presença não estabelece a elegibilidade atual de pagamento nem as regras de parcelamento. O agente deve consultar a política da loja para responder a essas questões.

**Atividade das promoções (25 promoções):**

- `is_active = 0`: 21
- `is_active = 1`: 4

Somente registros de promoções ativas devem ser considerados pela ferramenta de consulta de promoções, respeitando as regras de negócio da loja.

### 11.6 Ambiguidades investigadas nos dados

**O e-mail do cliente não é único.** O notebook encontrou dois registros de clientes com o mesmo endereço de e-mail não nulo:

- ID do cliente `11`: Bruno Carvalho Martins
- ID do cliente `43`: Bruno Martins

Portanto, o e-mail não deve ser tratado como uma chave única de cliente. A consulta de pedidos deve utilizar uma estratégia de identificação mais segura e não deve revelar informações do pedido de um cliente a outra pessoa. O conjunto de dados fornecido não oferece um mecanismo de verificação de identidade adequado para produção.

**Existem várias linhas de promoção para alguns produtos.** Os produtos `93`, `96` e `99` possuem dois registros de promoção cada. A verificação não encontrou nenhum produto com mais de uma promoção ativa neste conjunto de dados. Ainda assim, a aplicação deve filtrar por `is_active` e evitar seleções arbitrárias caso os dados passem a conter várias promoções ativas aplicáveis.

**A conversão de datas foi bem-sucedida nas colunas testadas.** Todos os 65 valores não nulos de `products.created_at` e os 20 valores de `orders.order_date` foram convertidos corretamente. Os 8 valores não nulos de `orders.estimated_delivery` também foram convertidos corretamente. As 12 datas restantes de entrega são nulas, não falhas de conversão.